# Martingales and Applications

| Difficulty | ███████░░░ 7/10 |
| :--- | :--- |
| Prerequisites | 09.03 |
| Optional | 09.04, 13.07 |
| Time | ~75 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/09_Advanced_Probability/07_martingales_and_applications.ipynb)

---

## 🎯 Learning Objective

Understand martingales as the mathematical framework for "fair games" and sequential
accumulation of information, master the optional stopping theorem and Azuma-Hoeffding
inequality, and see how SGD convergence is a supermartingale argument.

---

## 📐 Theory

`09.03` studied Markov chains — processes whose future depends on the past only through the
present state. This notebook studies **martingales** — processes whose *conditional expected*
future equals the present value. The distinction matters: a Markov chain's next state
distribution depends only on the current state; a martingale's *conditional mean* equals the
current value, but the distribution can depend on the entire history. Martingales are the
right language for SGD convergence (`04.02`), online learning, sequential testing, and the
Azuma-Hoeffding inequality (`09.04`'s Hoeffding bound generalized to dependent sequences).

### 1. Definition: martingales and filtrations

A **filtration** $\{\mathcal{F}_t\}_{t \ge 0}$ is a nested sequence of $\sigma$-algebras
(`09.01`): $\mathcal{F}_0 \subseteq \mathcal{F}_1 \subseteq \mathcal{F}_2 \subseteq \cdots$.
Intuitively, $\mathcal{F}_t$ encodes "all information available at time $t$." In ML practice:
$\mathcal{F}_t$ = the first $t$ mini-batches, or the first $t$ data points.

A sequence of random variables $\{M_t\}_{t \ge 0}$ **adapted** to $\{\mathcal{F}_t\}$ (i.e.,
$M_t$ is measurable with respect to $\mathcal{F}_t$ — knowing the information at time $t$
determines $M_t$'s value) is a **martingale** if:

1. $\mathbb{E}[|M_t|] < \infty$ for all $t$ (integrability), and
2. $\mathbb{E}[M_{t+1} \mid \mathcal{F}_t] = M_t$ for all $t$ (the **martingale property**).

The martingale property says: given everything you know up to time $t$, the *best prediction*
of $M_{t+1}$ is $M_t$ itself — a "fair game" where no strategy based on past information can
predict an upward or downward drift.

**Supermartingale / submartingale.** Replace the equality with an inequality:

- **Supermartingale:** $\mathbb{E}[M_{t+1} \mid \mathcal{F}_t] \le M_t$ — the process
  *decreases* in conditional expectation. ("Super" refers to the current value being *above*
  the conditional expectation of the next step, not to the process being "better" in any
  colloquial sense.)
- **Submartingale:** $\mathbb{E}[M_{t+1} \mid \mathcal{F}_t] \ge M_t$ — the process
  *increases* in conditional expectation.

**Doob's convergence theorem** (stated, not proved): If $\{M_t\}$ is a non-negative
supermartingale (or a submartingale bounded in $L^1$), then $M_t$ converges almost surely to a
finite random variable $M_\infty$. This is the theoretical engine behind SGD convergence
arguments (Section 5 below).

### 2. Key examples

**Random walk with zero-mean steps.** Let $X_1, X_2, \ldots$ be i.i.d. with
$\mathbb{E}[X_i] = 0$, and define $M_t = \sum_{i=1}^t X_i$, with $M_0 = 0$. Then:

$$\mathbb{E}[M_{t+1} \mid \mathcal{F}_t] = \mathbb{E}[M_t + X_{t+1} \mid \mathcal{F}_t]
= M_t + \mathbb{E}[X_{t+1}] = M_t,$$

since $X_{t+1}$ is independent of $\mathcal{F}_t$. This is the prototypical martingale and
the simplest model of a fair game. The paths we simulate below are instances of this.

**Cumulative SGD noise.** In stochastic gradient descent, the stochastic gradient
$g_t = \nabla f(w_t) + \xi_t$ where $\mathbb{E}[\xi_t \mid \mathcal{F}_t] = 0$ (unbiased
gradient). The cumulative noise $M_t = \sum_{i=1}^t \xi_i$ is a martingale — the noise terms
are zero-mean conditioned on the past. This martingale drives the "diffusion" of SGD around
the loss landscape.

**Likelihood ratio under the true distribution.** Given two hypotheses $H_0$ (density $p_0$)
and $H_1$ (density $p_1$), and data $X_1, X_2, \ldots$ drawn i.i.d. from $p_0$, the
likelihood ratio $L_t = \prod_{i=1}^t \frac{p_1(X_i)}{p_0(X_i)}$ is a martingale under
$H_0$:

$$\mathbb{E}_{H_0}[L_{t+1} \mid \mathcal{F}_t] = L_t \cdot \mathbb{E}_{H_0}\!\left[
\frac{p_1(X_{t+1})}{p_0(X_{t+1})}\right] = L_t \cdot \int \frac{p_1(x)}{p_0(x)}\,
p_0(x)\,dx = L_t \cdot 1 = L_t.$$

This is the foundation of the Sequential Probability Ratio Test (SPRT, Section 3).

**Product martingale.** Let $\mathbb{E}[X_i \mid \mathcal{F}_{i-1}] = 0$ and define
$M_t = \prod_{i=1}^t (1 + X_i)$, with $M_0 = 1$. Then
$\mathbb{E}[M_{t+1} \mid \mathcal{F}_t] = M_t \cdot \mathbb{E}[1+X_{t+1} \mid
\mathcal{F}_t] = M_t \cdot 1 = M_t$. This structure appears in online learning
(multiplicative weights / Hedge algorithm) and in wealth processes for sequential betting.

### 3. Optional stopping theorem (OST)

A **stopping time** $\tau$ with respect to $\{\mathcal{F}_t\}$ is a random time such that the
event $\{\tau \le t\}$ is $\mathcal{F}_t$-measurable for every $t$ — the decision to stop at
or before time $t$ depends only on information available at time $t$, with no peeking into the
future. Examples: "stop the first time the random walk exceeds $+5$" is a stopping time;
"stop one step *before* the random walk exceeds $+5$" is **not** (it requires knowing the
future value).

**Optional Stopping Theorem.** If $\{M_t\}$ is a martingale and $\tau$ is a stopping time
satisfying any one of the following sufficient conditions:

- (a) $\tau$ is bounded: $\tau \le N$ a.s. for some constant $N$, or
- (b) $\mathbb{E}[\tau] < \infty$ and increments are bounded:
  $|M_{t+1} - M_t| \le c$ a.s., or
- (c) the stopped process $\{M_{t \wedge \tau}\}$ is uniformly integrable,

then $\mathbb{E}[M_\tau] = \mathbb{E}[M_0]$.

The power of OST is that it lets you compute expected values at *random* times, not just
deterministic ones — the martingale's "fairness" is preserved even when you stop adaptively.

**Application: Wald's equation.** Let $X_1, X_2, \ldots$ be i.i.d. with
$\mathbb{E}[X_i] = \mu$ and let $\tau$ be a stopping time with $\mathbb{E}[\tau] < \infty$.
Define $M_t = \sum_{i=1}^t (X_i - \mu)$ — a martingale (centered i.i.d. sum). Applying OST:
$\mathbb{E}[M_\tau] = M_0 = 0$, i.e.,

$$\mathbb{E}\!\left[\sum_{i=1}^\tau X_i\right] = \mu \cdot \mathbb{E}[\tau].$$

This remarkable identity says the expected cumulative sum at a *random* stopping time equals
the per-step mean times the expected number of steps — even though $\tau$ depends on the
$X_i$'s in a potentially complex way.

**Application: Sequential Probability Ratio Test (SPRT).** To test $H_0: X_i \sim p_0$
versus $H_1: X_i \sim p_1$, the SPRT computes the log-likelihood ratio
$\Lambda_t = \sum_{i=1}^t \log\frac{p_1(X_i)}{p_0(X_i)}$ and stops the first time
$\Lambda_t$ exits the interval $(\log\frac{\beta}{1-\alpha},\,\log\frac{1-\beta}{\alpha})$,
where $\alpha, \beta$ are the desired type-I and type-II error rates. Since the likelihood
ratio $L_t = e^{\Lambda_t}$ is a martingale under $H_0$, OST gives the relationship between
the error probabilities and the stopping boundaries, and Wald's equation gives the expected
sample size.

### 4. Azuma-Hoeffding inequality

If $\{M_t\}_{t=0}^T$ is a martingale with **bounded differences**
$|M_t - M_{t-1}| \le c_t$ a.s. for each $t$, then for any $s > 0$:

$$P(|M_T - M_0| \ge s) \le 2\exp\!\left(-\frac{s^2}{2\sum_{t=1}^T c_t^2}\right).$$

**Proof sketch** (Chernoff method applied to the martingale). For any $\lambda > 0$:

1. By Markov's inequality: $P(M_T - M_0 \ge s) \le e^{-\lambda s}\,\mathbb{E}[e^{\lambda(M_T - M_0)}]$.
2. Decompose $M_T - M_0 = \sum_{t=1}^T D_t$ where $D_t = M_t - M_{t-1}$ are the martingale
   differences. The key step: by the tower property of conditional expectation and the
   martingale property ($\mathbb{E}[D_t \mid \mathcal{F}_{t-1}] = 0$),
   $$\mathbb{E}[e^{\lambda D_t} \mid \mathcal{F}_{t-1}] \le e^{\lambda^2 c_t^2/2}$$
   (Hoeffding's lemma applied to the zero-mean, $[-c_t, c_t]$-bounded increment $D_t$).
3. Iterating the tower property: $\mathbb{E}[e^{\lambda(M_T-M_0)}] \le \prod_{t=1}^T e^{\lambda^2 c_t^2/2} = e^{\lambda^2 \sum c_t^2/2}$.
4. Optimizing $\lambda = s / \sum c_t^2$ gives the one-sided bound. A union bound over both
   tails gives the factor of $2$.

**Connection to `09.04`.** Hoeffding's inequality from `09.04` is the special case where
$M_t = \sum_{i=1}^t (X_i - \mathbb{E}[X_i])$ with *independent* bounded increments — a
particular kind of martingale. Azuma-Hoeffding drops the independence requirement entirely;
bounded differences alone suffice. This is crucial for ML: training iterates are *dependent*
(each mini-batch's gradient depends on the current weights, which depend on all previous
mini-batches), so `09.04`'s i.i.d. Hoeffding bound doesn't directly apply, but
Azuma-Hoeffding does.

**McDiarmid's inequality as a corollary.** McDiarmid's bounded-differences inequality from
`09.04` — for a function $f(X_1,\ldots,X_n)$ satisfying
$\sup_{x_i, x_i'} |f(\ldots,x_i,\ldots) - f(\ldots,x_i',\ldots)| \le c_i$ — is obtained by
applying Azuma-Hoeffding to the **Doob martingale**
$M_t = \mathbb{E}[f(X_1,\ldots,X_n) \mid X_1,\ldots,X_t]$. The bounded-differences
condition on $f$ guarantees $|M_t - M_{t-1}| \le c_t$, and the Doob construction ensures
$\{M_t\}$ is a martingale. The CHALLENGE exercise below asks you to carry this out.

### 5. SGD as a supermartingale

Consider minimizing a convex, $L$-smooth function $f$ with stochastic gradient descent:
$w_{t+1} = w_t - \eta_t g_t$, where $\mathbb{E}[g_t \mid \mathcal{F}_t] = \nabla f(w_t)$
and $\mathbb{E}[\|g_t - \nabla f(w_t)\|^2 \mid \mathcal{F}_t] \le \sigma^2$. Under
standard assumptions (convexity, $L$-smoothness, step size $\eta_t$ small enough), one can
show:

$$\mathbb{E}[f(w_{t+1}) - f^* \mid \mathcal{F}_t] \le (f(w_t) - f^*)
- \eta_t\|\nabla f(w_t)\|^2 \left(1 - \frac{L\eta_t}{2}\right)
+ \frac{L\eta_t^2 \sigma^2}{2}.$$

When $\eta_t$ is small enough that $1 - L\eta_t/2 > 0$ and the gradient term dominates the
noise term (formally: when $\sum_t \eta_t = \infty$ and $\sum_t \eta_t^2 < \infty$, the
Robbins-Monro conditions), the sequence $V_t = f(w_t) - f^*$ satisfies:

$$\mathbb{E}[V_{t+1} \mid \mathcal{F}_t] \le V_t - a_t + b_t,$$

where $a_t \ge 0$ (the gradient descent progress) and $\sum_t b_t < \infty$ (the accumulated
noise, bounded because $\sum \eta_t^2 < \infty$). This is precisely the setting of the
**Robbins-Siegmund theorem**: if $\{V_t\}$ is a non-negative adapted process satisfying
$\mathbb{E}[V_{t+1} \mid \mathcal{F}_t] \le V_t - a_t + b_t$ with $\sum b_t < \infty$, then
$V_t \to V_\infty$ a.s. for some finite $V_\infty$, and $\sum_t a_t < \infty$.

The consequence: $f(w_t) \to f^*$ almost surely, and
$\sum_t \eta_t \|\nabla f(w_t)\|^2 < \infty$ (which, combined with $\sum \eta_t = \infty$,
forces $\liminf \|\nabla f(w_t)\| = 0$). This is the **formal justification for SGD
convergence** — not a heuristic argument about "following the gradient downhill," but a
rigorous supermartingale convergence theorem. It connects `04.02`'s practical SGD
implementations, `09.04`'s concentration inequalities (which give convergence *rates*), and
`13.07`'s stability perspective (which gives generalization bounds) into a single theoretical
framework.

---

In [ ]:
# Setup
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Three visualizations make the core concepts tangible: (1) 500 random walk paths with the
±Azuma-Hoeffding envelope overlaid — the envelope tightens the concentration bound, and
almost no path escapes it; (2) the optional stopping theorem verified empirically — a
random walk stopped at a hitting time still has $\mathbb{E}[M_\tau] = M_0$ across thousands
of trials; (3) SGD on a convex function showing supermartingale behavior — individual paths
spike upward, but the *conditional mean* always decreases.

In [ ]:
rng = np.random.default_rng(42)

# ── Panel 1: Random walk paths + Azuma-Hoeffding envelope ──
n_paths, n_steps = 500, 200
# Rademacher steps: +1 or -1 with equal probability → bounded differences |D_t| ≤ 1
steps = rng.choice([-1, 1], size=(n_paths, n_steps))
walks = np.cumsum(steps, axis=1)
walks = np.hstack([np.zeros((n_paths, 1)), walks])  # prepend M_0 = 0

time_axis = np.arange(n_steps + 1)
# Azuma-Hoeffding: P(|M_T| >= s) <= 2*exp(-s^2/(2T)) when c_t=1 for all t
# For a 99% envelope: 2*exp(-s^2/(2T)) = 0.01 → s = sqrt(2T * ln(200))
alpha_env = 0.01
envelope = np.sqrt(2 * time_axis * np.log(2.0 / alpha_env))
envelope[0] = 0

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

ax = axes[0]
for i in range(n_paths):
    ax.plot(time_axis, walks[i], color='#4C72B0', alpha=0.03, linewidth=0.5)
ax.plot(time_axis, envelope, color='#C44E52', linewidth=2.5, linestyle='--',
        label=f'Azuma-Hoeffding 99% envelope')
ax.plot(time_axis, -envelope, color='#C44E52', linewidth=2.5, linestyle='--')
ax.set_xlabel('Time step t')
ax.set_ylabel('$M_t$')
ax.set_title('500 random walks + Azuma-Hoeffding envelope\n'
             '(bounded differences $|D_t| \\leq 1$)')
ax.legend(fontsize=9, loc='upper left')

# Count exceedances at the final time step
final_exceed = np.mean(np.abs(walks[:, -1]) >= envelope[-1])
ax.text(0.98, 0.02, f'Empirical exceedance at T={n_steps}: {final_exceed:.1%}\n'
        f'Azuma-Hoeffding bound: {alpha_env:.0%}',
        transform=ax.transAxes, fontsize=8, ha='right', va='bottom',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='wheat', alpha=0.8))

# ── Panel 2: Optional Stopping Theorem verification ──
ax = axes[1]
n_trials = 10_000
barrier = 10
max_t = 2000  # safety cap
stopped_values = []
stopping_times = []

for _ in range(n_trials):
    M = 0.0
    for t in range(1, max_t + 1):
        M += rng.choice([-1.0, 1.0])
        if abs(M) >= barrier:
            break
    stopped_values.append(M)
    stopping_times.append(t)

stopped_values = np.array(stopped_values)
stopping_times = np.array(stopping_times)

ax.hist(stopped_values, bins=50, density=True, color='#4C72B0', alpha=0.7,
        edgecolor='white')
ax.axvline(np.mean(stopped_values), color='#C44E52', linewidth=2.5,
           label=f'$E[M_\\tau] = {np.mean(stopped_values):.3f}$ (should be $\\approx 0$)')
ax.axvline(0, color='black', linewidth=1, linestyle='--', label='$M_0 = 0$')
ax.set_xlabel('$M_\\tau$ (value at stopping time)')
ax.set_ylabel('Density')
ax.set_title(f'OST verification: {n_trials:,} trials,\n'
             f'stop at $|M_t| \\geq {barrier}$')
ax.legend(fontsize=9)

# ── Panel 3: SGD supermartingale behavior ──
ax = axes[2]
# Minimize f(w) = 0.5*w^2 (convex, L-smooth with L=1) via SGD with noisy gradients
n_sgd_runs, n_sgd_steps = 100, 300
w0 = 5.0
f_star = 0.0
sigma_noise = 1.0

all_gaps = np.zeros((n_sgd_runs, n_sgd_steps + 1))
for run in range(n_sgd_runs):
    w = w0
    all_gaps[run, 0] = 0.5 * w**2 - f_star
    for t in range(1, n_sgd_steps + 1):
        eta = 0.1 / (1 + 0.01 * t)  # decaying step size satisfying Robbins-Monro
        grad_true = w  # ∇f(w) = w for f(w)=0.5*w^2
        noise = rng.normal(0, sigma_noise)
        g = grad_true + noise  # unbiased stochastic gradient
        w = w - eta * g
        all_gaps[run, t] = 0.5 * w**2 - f_star

for run in range(min(50, n_sgd_runs)):
    ax.plot(all_gaps[run], color='#4C72B0', alpha=0.08, linewidth=0.5)
mean_gap = all_gaps.mean(axis=0)
ax.plot(mean_gap, color='#C44E52', linewidth=2.5, label='$E[f(w_t) - f^*]$ (mean over runs)')
ax.set_xlabel('SGD step $t$')
ax.set_ylabel('$f(w_t) - f^*$')
ax.set_title('SGD supermartingale: $f(w_t)-f^*$ decreases\n'
             'in expectation (individual paths can spike)')
ax.set_yscale('log')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"OST check: E[M_\u03c4] = {np.mean(stopped_values):.4f} (M_0 = 0, should be \u2248 0)")
print(f"Wald's equation check: E[\u03c4] = {np.mean(stopping_times):.1f}, "
      f"barrier\u00b2 = {barrier**2} (for \u00b11 walk hitting \u00b1{barrier}, E[\u03c4] = {barrier}\u00b2 = {barrier**2})")

## 🐍 Implementation from Scratch

Four implementations, each verifying a core concept numerically: (1) random walk martingale
property, (2) Azuma-Hoeffding bound versus empirical exceedance, (3) the Sequential
Probability Ratio Test with OST-derived guarantees, and (4) SGD as a supermartingale on
logistic regression.

In [ ]:
rng = np.random.default_rng(0)

# ═════════════════════════════════════════════════════════════════
# 1. Random walk: verify the martingale property empirically
# ═════════════════════════════════════════════════════════════════
def verify_martingale_property(n_paths=50_000, n_steps=100, step_fn=None, rng=None):
    """Verify E[M_{t+1} | F_t] approx M_t for a random walk martingale.
    Returns (max_abs_deviation, time_of_max_deviation).
    """
    if step_fn is None:
        step_fn = lambda size: rng.choice([-1.0, 1.0], size=size)
    steps = step_fn(size=(n_paths, n_steps))
    walks = np.cumsum(steps, axis=1)
    walks = np.hstack([np.zeros((n_paths, 1)), walks])

    # At each t, E[M_{t+1} - M_t] should be approx 0 for a martingale
    max_dev = 0.0
    worst_t = 0
    for t in range(0, min(n_steps, 50)):
        dev = abs(np.mean(walks[:, t + 1] - walks[:, t]))
        if dev > max_dev:
            max_dev = dev
            worst_t = t
    return max_dev, worst_t

dev, t = verify_martingale_property(rng=rng)
print(f"Martingale property check (random walk, 50k paths):")
print(f"  Max |E[M_{{t+1}} - M_t]| = {dev:.5f} at t={t} (should be \u2248 0)")
print()

# ═════════════════════════════════════════════════════════════════
# 2. Azuma-Hoeffding: compare bound to empirical exceedance
# ═════════════════════════════════════════════════════════════════
def azuma_hoeffding_bound(s, c_squared_sum):
    """P(|M_T - M_0| >= s) <= 2*exp(-s^2 / (2 * sum(c_t^2)))."""
    return 2.0 * np.exp(-s**2 / (2.0 * c_squared_sum))

n_paths_ah, T = 100_000, 200
steps_ah = rng.choice([-1.0, 1.0], size=(n_paths_ah, T))
final_values = steps_ah.sum(axis=1)  # M_T - M_0

print(f"Azuma-Hoeffding bound vs empirical exceedance (T={T}, c_t=1):")
print(f"{'s':>5}  {'AH bound':>12}  {'Empirical P':>12}  {'Bound holds?':>14}")
for s in [5, 10, 15, 20, 25]:
    bound = azuma_hoeffding_bound(s, T)  # sum(c_t^2) = T when c_t = 1
    empirical = np.mean(np.abs(final_values) >= s)
    print(f"{s:5d}  {bound:12.6f}  {empirical:12.6f}  {'\u2713' if empirical <= bound else '\u2717':>14}")
print()

# ═════════════════════════════════════════════════════════════════
# 3. SPRT for two Gaussians: OST in action
# ═════════════════════════════════════════════════════════════════
def sprt_two_gaussians(mu0, mu1, sigma, alpha, beta, rng, true_hyp='H0', max_steps=10000):
    """Sequential Probability Ratio Test for N(mu0, sigma^2) vs N(mu1, sigma^2).
    Returns (decision, stopping_time, log_likelihood_ratio_at_stop).
    """
    lower = np.log(beta / (1 - alpha))
    upper = np.log((1 - beta) / alpha)
    true_mu = mu0 if true_hyp == 'H0' else mu1

    Lambda = 0.0
    for t in range(1, max_steps + 1):
        x = rng.normal(true_mu, sigma)
        # log(p1(x)/p0(x)) for Gaussians with equal variance:
        # = (mu1-mu0)/sigma^2 * (x - (mu0+mu1)/2)
        Lambda += (mu1 - mu0) / sigma**2 * (x - (mu0 + mu1) / 2)
        if Lambda >= upper:
            return 'H1', t, Lambda
        elif Lambda <= lower:
            return 'H0', t, Lambda
    return 'undecided', max_steps, Lambda

mu0, mu1, sigma = 0.0, 1.0, 1.0
alpha, beta = 0.05, 0.05
n_trials = 5000

# Under H0
results_h0 = [sprt_two_gaussians(mu0, mu1, sigma, alpha, beta, rng, 'H0')
              for _ in range(n_trials)]
decisions_h0, times_h0, _ = zip(*results_h0)
type1_error = np.mean([d == 'H1' for d in decisions_h0])
mean_stop_h0 = np.mean(times_h0)

# Under H1
results_h1 = [sprt_two_gaussians(mu0, mu1, sigma, alpha, beta, rng, 'H1')
              for _ in range(n_trials)]
decisions_h1, times_h1, _ = zip(*results_h1)
type2_error = np.mean([d == 'H0' for d in decisions_h1])
mean_stop_h1 = np.mean(times_h1)

# Wald's expected sample size under H0
kl_h0 = -(mu1 - mu0)**2 / (2 * sigma**2)
wald_expected_h0 = (alpha * np.log(beta/(1-alpha)) +
                    (1-alpha) * np.log((1-beta)/alpha)) / kl_h0

print(f"SPRT for N({mu0},{sigma}\u00b2) vs N({mu1},{sigma}\u00b2), \u03b1=\u03b2={alpha}:")
print(f"  Under H0: Type-I error = {type1_error:.3f} (target \u2264 {alpha})")
print(f"            E[\u03c4] = {mean_stop_h0:.1f} (Wald approx: {abs(wald_expected_h0):.1f})")
print(f"  Under H1: Type-II error = {type2_error:.3f} (target \u2264 {beta})")
print(f"            E[\u03c4] = {mean_stop_h1:.1f}")
print()

# ═════════════════════════════════════════════════════════════════
# 4. SGD supermartingale on logistic regression
# ═════════════════════════════════════════════════════════════════
def logistic_loss(w, X, y):
    """Average logistic loss (convex in w)."""
    z = y * (X @ w)
    return np.mean(np.logaddexp(0, -z))

def logistic_grad(w, X, y):
    """Gradient of average logistic loss."""
    z = y * (X @ w)
    s = -y / (1 + np.exp(z))
    return (X.T @ s) / len(y)

# Generate a linearly separable dataset
d = 5
n_data = 500
w_true = rng.standard_normal(d)
X_data = rng.standard_normal((n_data, d))
y_data = np.sign(X_data @ w_true)
y_data[y_data == 0] = 1.0

# Find f* by running many steps of full-batch GD
w_opt = np.zeros(d)
for _ in range(5000):
    w_opt -= 0.5 * logistic_grad(w_opt, X_data, y_data)
f_star = logistic_loss(w_opt, X_data, y_data)

# SGD: track f(w_t) - f* and verify supermartingale property
n_sgd_runs, n_sgd_steps = 100, 500
batch_size = 32
gaps = np.zeros((n_sgd_runs, n_sgd_steps + 1))

for run in range(n_sgd_runs):
    w = np.zeros(d)
    gaps[run, 0] = logistic_loss(w, X_data, y_data) - f_star
    for t in range(1, n_sgd_steps + 1):
        eta = 0.5 / (1 + 0.005 * t)
        idx = rng.choice(n_data, size=batch_size, replace=False)
        g = logistic_grad(w, X_data[idx], y_data[idx])
        w = w - eta * g
        gaps[run, t] = logistic_loss(w, X_data, y_data) - f_star

# Verify supermartingale: E[V_{t+1}] should be <= E[V_t] on average
mean_gaps = gaps.mean(axis=0)
n_supermartingale = np.sum(mean_gaps[1:] <= mean_gaps[:-1] + 1e-10)
print(f"SGD supermartingale check (logistic regression, {n_sgd_runs} runs):")
print(f"  E[V_{{t+1}}] \u2264 E[V_t] holds for {n_supermartingale}/{n_sgd_steps} steps "
      f"({n_supermartingale/n_sgd_steps:.1%})")
print(f"  Initial gap: E[V_0] = {mean_gaps[0]:.4f}")
print(f"  Final gap:   E[V_T] = {mean_gaps[-1]:.6f}")
print(f"  f* = {f_star:.6f}")

## 🤖 Why This Matters for AI

**SGD convergence is a supermartingale argument.** The standard proof that SGD converges
(Robbins & Monro, 1951; the foundational result behind every modern training run) is not a
gradient-descent-style "$f$ decreases each step" argument — it's a *supermartingale
convergence* argument. The suboptimality gap $f(w_t) - f^*$ doesn't decrease every step
(individual gradient samples are noisy), but it decreases *in conditional expectation*, and
that's enough for the Robbins-Siegmund theorem to guarantee convergence. This is the formal
bridge from `04.02`'s practical SGD implementations to `13.07`'s generalization theory.

**Online learning and regret analysis.** In adversarial online learning (Prediction with
Expert Advice, multi-armed bandits), the regret — the difference between your cumulative loss
and the best fixed action in hindsight — is controlled by martingale concentration bounds.
The product martingale from Section 2 appears explicitly in the analysis of the multiplicative
weights / Hedge algorithm.

**Confidence sequences (always-valid inference).** Classical confidence intervals are valid at
a *fixed* sample size; if you peek at intermediate results and stop early, coverage guarantees
break. **Confidence sequences** — confidence intervals that are simultaneously valid at *all*
sample sizes — are constructed using martingale inequalities (specifically, the method of
mixtures and Ville's inequality, a continuous-time extension of OST). These are the
mathematical backbone of modern A/B testing platforms that allow continuous monitoring.

**Concentration for dependent data.** The Azuma-Hoeffding inequality is the workhorse
concentration bound for ML settings where `09.04`'s i.i.d. assumptions fail. Training
iterates are dependent (each mini-batch's contribution depends on the current model, which
depends on all previous mini-batches); Azuma-Hoeffding handles this because it only requires
bounded differences, not independence. McDiarmid's inequality — used throughout statistical
learning theory (`13.07`) to prove uniform convergence — is a direct corollary of
Azuma-Hoeffding via the Doob martingale construction.

The cell below demonstrates the SGD supermartingale property on logistic regression at scale,
tracking both the raw $f(w_t) - f^*$ paths and the conditional-expectation decrease across
100 independent seeds.

In [ ]:
# AI Application: SGD supermartingale on logistic regression, 100 seeds
# Tracks f(w_t) - f* and empirically verifies E[V_{t+1} | F_t] <= V_t.

rng_ai = np.random.default_rng(123)

# --- Data generation ---
d, n_data = 10, 1000
w_true = rng_ai.standard_normal(d)
X = rng_ai.standard_normal((n_data, d))
y = np.sign(X @ w_true)
y[y == 0] = 1.0

# --- Find f* via full-batch GD ---
w_opt = np.zeros(d)
for _ in range(10_000):
    z = y * (X @ w_opt)
    grad = (X.T @ (-y / (1 + np.exp(z)))) / n_data
    w_opt -= 0.3 * grad
f_star = np.mean(np.logaddexp(0, -y * (X @ w_opt)))

# --- SGD across 100 seeds ---
n_seeds, T = 100, 10_000
batch_size = 64
# Store gaps at log-spaced checkpoints to save memory
checkpoints = np.unique(np.geomspace(1, T, 200).astype(int))
checkpoints = np.concatenate([[0], checkpoints])
gap_matrix = np.zeros((n_seeds, len(checkpoints)))

for seed in range(n_seeds):
    rng_s = np.random.default_rng(seed)
    w = np.zeros(d)
    cp_idx = 0
    if checkpoints[cp_idx] == 0:
        gap_matrix[seed, cp_idx] = np.mean(np.logaddexp(0, -y * (X @ w))) - f_star
        cp_idx += 1
    for t in range(1, T + 1):
        eta = 1.0 / (1 + 0.001 * t)
        idx = rng_s.choice(n_data, size=batch_size, replace=False)
        z_batch = y[idx] * (X[idx] @ w)
        g = (X[idx].T @ (-y[idx] / (1 + np.exp(z_batch)))) / batch_size
        w = w - eta * g
        if cp_idx < len(checkpoints) and t == checkpoints[cp_idx]:
            gap_matrix[seed, cp_idx] = np.mean(np.logaddexp(0, -y * (X @ w))) - f_star
            cp_idx += 1

mean_gap = gap_matrix.mean(axis=0)
std_gap = gap_matrix.std(axis=0)
p10 = np.percentile(gap_matrix, 10, axis=0)
p90 = np.percentile(gap_matrix, 90, axis=0)

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

ax = axes[0]
for seed in range(min(30, n_seeds)):
    ax.plot(checkpoints, gap_matrix[seed], color='#4C72B0', alpha=0.1, linewidth=0.5)
ax.plot(checkpoints, mean_gap, color='#C44E52', linewidth=2.5,
        label='$E[f(w_t)-f^*]$ (mean)')
ax.fill_between(checkpoints, p10, p90, color='#C44E52', alpha=0.15,
                label='10th\u201390th percentile')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('SGD step $t$')
ax.set_ylabel('$f(w_t) - f^*$')
ax.set_title(f'SGD supermartingale: logistic regression\n'
             f'd={d}, n={n_data}, {n_seeds} seeds, {T:,} steps')
ax.legend(fontsize=9)

# --- Verify supermartingale property: E[V_{t+1}] <= E[V_t] ---
ax = axes[1]
deltas = mean_gap[1:] - mean_gap[:-1]
n_decrease = np.sum(deltas <= 1e-12)
colors = ['#55A868' if d <= 1e-12 else '#C44E52' for d in deltas]
ax.bar(range(len(deltas)), deltas, color=colors, width=1.0, alpha=0.7)
ax.axhline(0, color='black', linewidth=1)
ax.set_xlabel('Checkpoint index')
ax.set_ylabel('$E[V_{t+1}] - E[V_t]$')
ax.set_title(f'Supermartingale check: {n_decrease}/{len(deltas)} intervals decrease\n'
             f'(green = decrease, red = increase)')

plt.tight_layout()
plt.show()

print(f"\nSGD supermartingale summary ({n_seeds} seeds, {T:,} steps):")
print(f"  Mean initial gap:   {mean_gap[0]:.4f}")
print(f"  Mean final gap:     {mean_gap[-1]:.6f}")
print(f"  E[V_{{t+1}}] \u2264 E[V_t]: {n_decrease}/{len(deltas)} checkpoints "
      f"({n_decrease/len(deltas):.1%})")
print(f"  The few violations are small upward fluctuations from finite-sample averaging \u2014")
print(f"  the supermartingale property holds in expectation, not pathwise.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. Verify $\mathbb{E}[M_{t+1} \mid \mathcal{F}_t] = M_t$ for both the random walk
   $M_t = \sum_{i=1}^t X_i$ (with $X_i$ Rademacher, i.e. $\pm 1$ equally likely) and the
   product martingale $M_t = \prod_{i=1}^t (1 + X_i)$ (with $X_i$ uniform on $[-0.5, 0.5]$,
   so $\mathbb{E}[X_i] = 0$). Do this by direct calculation on paper, then verify
   numerically with 50,000 simulated paths.

<details>
<summary>💡 Solution</summary>

**Random walk:** $\mathbb{E}[M_{t+1} \mid \mathcal{F}_t] = \mathbb{E}[M_t + X_{t+1} \mid
\mathcal{F}_t] = M_t + \mathbb{E}[X_{t+1}] = M_t + 0 = M_t$. ✓

**Product martingale:** $\mathbb{E}[M_{t+1} \mid \mathcal{F}_t] = \mathbb{E}[M_t(1+X_{t+1})
\mid \mathcal{F}_t] = M_t \cdot \mathbb{E}[1+X_{t+1}] = M_t \cdot 1 = M_t$. ✓

Numerically: simulate 50,000 paths of each. At each time $t$, compute
$\text{mean}(M_{t+1} - M_t)$ — this should be $\approx 0$ for both.
```python
rng = np.random.default_rng(0)
# Random walk
steps = rng.choice([-1.0, 1.0], size=(50000, 100))
walks = np.cumsum(steps, axis=1)
diffs = np.diff(walks, axis=1)
print(f"Random walk max |E[D_t]|: {np.max(np.abs(diffs.mean(axis=0))):.5f}")

# Product martingale
X = rng.uniform(-0.5, 0.5, size=(50000, 100))
prod = np.cumprod(1 + X, axis=1)
ratios = prod[:, 1:] / prod[:, :-1] - 1  # M_{t+1}/M_t - 1 = X_{t+1}
print(f"Product martingale max |E[X_{{t+1}}]|: {np.max(np.abs(ratios.mean(axis=0))):.5f}")
```
Both maxima should be $< 0.01$, confirming the martingale property empirically.

</details>

### 💭 UNDERSTAND
2. Why is a *super*martingale called "super" when it *decreases* in expectation? The name
   seems backwards at first glance.

<details>
<summary>💡 Solution</summary>

"Super" refers to the current value $M_t$ being *above* ("super" = Latin for "above") the
conditional expectation $\mathbb{E}[M_{t+1} \mid \mathcal{F}_t]$. The defining inequality is
$M_t \ge \mathbb{E}[M_{t+1} \mid \mathcal{F}_t]$ — the present value *exceeds* (is
"super" to) the expected future. This is consistent with the convention in analysis where
"superharmonic" means the function value at a point exceeds its average over a neighborhood.
The name describes the *relationship* between the current value and the conditional
expectation, not the direction of the process.

</details>

### ✏️ DERIVE
3. Prove Wald's equation for a random walk with i.i.d. increments using OST. Specifically:
   let $X_1, X_2, \ldots$ be i.i.d. with $\mathbb{E}[X_i] = \mu$ and $\text{Var}(X_i) =
   \sigma^2 < \infty$. Let $\tau$ be a stopping time with $\mathbb{E}[\tau] < \infty$.
   Show that $\mathbb{E}[\sum_{i=1}^\tau X_i] = \mu \cdot \mathbb{E}[\tau]$.

<details>
<summary>💡 Solution outline</summary>

Define $M_t = \sum_{i=1}^t (X_i - \mu)$. This is a martingale:
$\mathbb{E}[M_{t+1} \mid \mathcal{F}_t] = M_t + \mathbb{E}[X_{t+1} - \mu] = M_t$.

The increments $D_t = X_t - \mu$ satisfy $|D_t| \le |X_t| + |\mu|$, and since
$\mathbb{E}[|X_i|] < \infty$ (implied by finite variance), the increments are integrable.
With $\mathbb{E}[\tau] < \infty$ and integrable bounded increments, condition (b) of OST
applies (or more precisely, Wald's original argument uses the dominated-convergence
approach). OST gives:

$\mathbb{E}[M_\tau] = \mathbb{E}[M_0] = 0$.

Expanding: $\mathbb{E}[\sum_{i=1}^\tau (X_i - \mu)] = 0$, i.e.,
$\mathbb{E}[\sum_{i=1}^\tau X_i] = \mu \cdot \mathbb{E}[\tau]$. $\square$

</details>

### 🐍 IMPLEMENT
4. Implement the SPRT for testing $N(0, 1)$ versus $N(0.5, 1)$ with
   $\alpha = \beta = 0.05$. Run 5,000 trials under each hypothesis. Verify: (a) the
   type-I error is $\le \alpha$, (b) the type-II error is $\le \beta$, and (c) the empirical
   $\mathbb{E}[\tau]$ under $H_0$ is close to Wald's approximation.

<details>
<summary>✅ How to know you're right</summary>

Use the `sprt_two_gaussians` function from the Implementation cell (or your own
re-implementation). With $\mu_0=0, \mu_1=0.5, \sigma=1, \alpha=\beta=0.05$:
- Type-I error should be $\lesssim 0.05$ (often around $0.03$–$0.05$).
- Type-II error should be $\lesssim 0.05$.
- $\mathbb{E}[\tau]$ under $H_0$: Wald's approximation gives
  $\approx \frac{\alpha \log(\beta/(1-\alpha)) + (1-\alpha)\log((1-\beta)/\alpha)}
  {-(\mu_1-\mu_0)^2/(2\sigma^2)} \approx 22$. Empirical values should be in the range
  $20$–$25$.

If your type-I error significantly exceeds $\alpha$, check that your log-likelihood ratio
formula is correct — the most common bug is getting the sign wrong (comparing $p_1/p_0$
instead of $p_0/p_1$, or vice versa).

</details>

### 🤖 APPLY
5. Run SGD on logistic regression with the setup from the Implementation cell. Track
   $f(w_t) - f^*$ over 10,000 steps across 100 random seeds. At each step $t$, compute the
   empirical $\mathbb{E}[f(w_{t+1}) - f^* \mid \mathcal{F}_t]$ by averaging over seeds, and
   verify it's $\le f(w_t) - f^*$ (the supermartingale property). Report the fraction of
   steps where this holds.

<details>
<summary>✅ What you should observe</summary>

The supermartingale property $\mathbb{E}[V_{t+1}] \le \mathbb{E}[V_t]$ should hold for
$\ge 90\%$ of steps, with violations concentrated at early steps (where step sizes are large
and noise dominates) and appearing as small upward blips, not sustained increases. The overall
trend is unambiguously downward. The AI Application cell above runs exactly this experiment
— you should reproduce it independently and get qualitatively similar results.

A subtle point: we're checking $\mathbb{E}[V_{t+1}] \le \mathbb{E}[V_t]$ (unconditional
expectations, averaged over seeds), which is *implied by* but *weaker than* the true
supermartingale property $\mathbb{E}[V_{t+1} \mid \mathcal{F}_t] \le V_t$ (which holds
pathwise in conditional expectation). The unconditional version is what we can check
empirically with finite seeds; the conditional version is the theoretical guarantee.

</details>

### 🚀 CHALLENGE
6. The **Doob martingale** $M_t = \mathbb{E}[f(X_1,\ldots,X_n) \mid X_1,\ldots,X_t]$ has
   bounded differences $|M_t - M_{t-1}| \le c_t$ whenever $f$ satisfies the
   bounded-differences property
   $\sup_{x_i, x_i'} |f(\ldots,x_i,\ldots) - f(\ldots,x_i',\ldots)| \le c_i$ from
   `09.04`'s McDiarmid's inequality. Derive McDiarmid's inequality as a corollary of
   Azuma-Hoeffding applied to this Doob martingale.

<details>
<summary>🔍 What a strong answer covers</summary>

**Step 1:** Define the Doob martingale. Let $X_1, \ldots, X_n$ be independent (not
necessarily identically distributed). Define:
$M_0 = \mathbb{E}[f(X_1,\ldots,X_n)]$,
$M_t = \mathbb{E}[f(X_1,\ldots,X_n) \mid X_1,\ldots,X_t]$ for $t=1,\ldots,n$.
Note $M_n = f(X_1,\ldots,X_n)$ (conditioning on all variables = knowing the value).

**Step 2:** Verify $\{M_t\}$ is a martingale. By the tower property:
$\mathbb{E}[M_{t+1} \mid \mathcal{F}_t] = \mathbb{E}[\mathbb{E}[f \mid X_1,\ldots,X_{t+1}]
\mid X_1,\ldots,X_t] = \mathbb{E}[f \mid X_1,\ldots,X_t] = M_t$. ✓

**Step 3:** Bound the differences. $M_t - M_{t-1} = \mathbb{E}[f \mid X_1,\ldots,X_t]
- \mathbb{E}[f \mid X_1,\ldots,X_{t-1}]$. Since $f$ satisfies bounded differences with
constant $c_t$ in coordinate $t$, and since $M_t - M_{t-1}$ is obtained by averaging $f$ over
all coordinates except $1,\ldots,t$ while varying only $X_t$, we get
$|M_t - M_{t-1}| \le c_t$ a.s. (the averaging can only reduce the range, not increase it).

**Step 4:** Apply Azuma-Hoeffding to $\{M_t\}$:
$P(|M_n - M_0| \ge s) \le 2\exp(-s^2 / (2\sum_{t=1}^n c_t^2))$.
Since $M_n = f(X_1,\ldots,X_n)$ and $M_0 = \mathbb{E}[f(X_1,\ldots,X_n)]$, this is exactly
McDiarmid's inequality: $P(|f(X) - \mathbb{E}[f(X)]| \ge s) \le 2\exp(-2s^2 / \sum c_i^2)$.

(Note: the standard form of McDiarmid has $2s^2$ in the numerator rather than $s^2$ due to a
different convention for the bounded-differences constant — some sources define $c_i$ as a
half-range rather than a full range. The structural result is the same: McDiarmid IS
Azuma-Hoeffding applied to the Doob martingale.)

</details>

---

## 📚 Further Reading
- Williams, *Probability with Martingales* (1991) — the classic introduction
- Wainwright, *High-Dimensional Statistics*, Ch. 5 (2019) — Azuma-Hoeffding and martingale
  concentration in a modern ML context
- Robbins & Siegmund, "A convergence theorem for non-negative almost supermartingales" (1971)
  — the supermartingale convergence theorem used in SGD analysis
- Howard et al., "Time-uniform, nonparametric, nonasymptotic confidence sequences" (2021)
  — the modern theory of confidence sequences built on martingale inequalities

---

*Next notebook: [`09.08` Markov Decision Processes](08_markov_decision_processes.ipynb)*